# Módulo 3 · Notebook 3 — Base vectorial: Historia de los Mundiales

Indexamos el PDF **Historia de los mundiales (1930–2010)** en ChromaDB y lo guardamos de forma persistente en `data/vector_db_2/`.

**Requisitos:** Ollama en marcha con `nomic-embed-text:latest`.

## 1. Rutas y setup

In [ ]:
import warnings
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

BASE_DIR = Path("../../").resolve()
PDF_PATH = BASE_DIR / "data" / "Football" / "Historia-de-los-mundiales-(1930-2010).pdf"
VECTOR_DIR = str(BASE_DIR / "data" / "vector_db_2")
COLLECTION_NAME = "mundiales_football"

Path(VECTOR_DIR).mkdir(parents=True, exist_ok=True)

print(f"📄 PDF    → {PDF_PATH}")
print(f"📁 Chroma → {VECTOR_DIR}")
print(f"   Existe PDF: {PDF_PATH.exists()}")

## 2. Cargar el PDF

In [ ]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF: {PDF_PATH}")

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"📄 Páginas cargadas: {len(pages)}")
print(f"\n🔍 Vista previa (página 1):\n{pages[0].page_content[:400]}...")

## 3. Dividir en chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len,
)

documents = text_splitter.split_documents(pages)

print(f"✂️  Chunks generados: {len(documents)}")
print(f"\n🔍 Ejemplo (chunk #5):\n{documents[5].page_content[:400]}...")

## 4. Crear y persistir ChromaDB

La primera ejecución puede tardar varios minutos (embedding de todos los chunks).

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=VECTOR_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"✅ Vector store guardado en: {VECTOR_DIR}")
print(f"   Colección: {COLLECTION_NAME}")
print(f"   Documentos indexados: {vectorstore._collection.count()}")

## 5. Cargar desde disco (sin re-indexar)

Si ya ejecutaste la celda anterior, usa esto en sesiones posteriores:

In [ ]:
# embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
# vectorstore = Chroma(
#     persist_directory=VECTOR_DIR,
#     embedding_function=embeddings,
#     collection_name=COLLECTION_NAME,
# )
# print(f"✅ Cargado desde disco: {vectorstore._collection.count()} docs")

## 6. Prueba de búsqueda semántica

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "¿Quién ganó el Mundial de 1978 y dónde se jugó?"
docs = retriever.invoke(query)

print(f"Pregunta: {query}\n")
for i, doc in enumerate(docs, 1):
    page = doc.metadata.get("page", "?")
    print(f"--- Resultado {i} (pág. {page + 1}) ---")
    print(doc.page_content[:350].strip())
    print()